# CodeMix Intelligence — mT5-small Hinglish-to-English Baseline
## End-to-End Reproducible Fine-Tuning on Google Colab GPU (NVIDIA T4)

```
========================================================================================
Project:              CodeMix Intelligence (Multilingual Code-Mixed NLP Platform)
Model Target:         google/mt5-small (~300M parameters, Multilingual T5)
Task:                 Romanized Hinglish Sentence -> Natural English Translation
Dataset:              PHINC (Parallel Hinglish Social Media Corpus, W-NUT 2020)
License:              Creative Commons Attribution 4.0 International (CC BY 4.0)
Translation Source:   Human-Translated Ground Truth
Target Environment:   Google Colab Single GPU (NVIDIA T4 16GB VRAM recommended)
Development Branch:   feature/mt5-translation-v2
Protected Baseline:   v1.0.0 (MuRIL Token-Level Language Identification Engine)
========================================================================================
```

### Strict Scientific Integrity & Anti-Leakage Safeguards:
1. **LOCKED TEST SET ISOLATION:** The project test split (`phinc_test.jsonl`, 1,351 records) is cryptographically locked and **strictly excluded** from this training environment. The training bundle contains ONLY `phinc_train.jsonl` (10,802 pairs) and `phinc_validation.jsonl` (1,350 pairs).
2. **VALIDATION-ONLY SELECTION:** All checkpoint selection, early stopping, and hyperparameter tuning are guided strictly by validation metrics (SacreBLEU / chrF / validation loss). Test evaluation is never performed inside Colab.
3. **ROMANIZED HINGLISH SCOPE:** The training data consists primarily of Romanized Hindi-English code-mixed social media text in Latin script. This notebook does not train Telugu translation, and no Telugu or full trilingual translation claims are made.


---
## SECTION A — Introduction and Safeguards

### 1. Project Background
Code-mixing is the linguistic phenomenon where speakers effortlessly alternate between two or more languages within a single utterance or conversation (e.g., *"Meeting ke baad call karo"* $\rightarrow$ *"Call me after the meeting"*).

In Version 1 of CodeMix Intelligence, we built and frozen a token-level Language Identification (LID) engine using `google/muril-base-cased` (preserved at tag `v1.0.0`). In Version 2, we build the complementary translation component using **`google/mt5-small`**, Google's multilingual sequence-to-sequence transformer pretrained on mC4 covering 101 languages.

### 2. Dataset Split Governance
- **Training Set (`phinc_train.jsonl`):** 10,802 records — Used for gradient descent fine-tuning.
- **Validation Set (`phinc_validation.jsonl`):** 1,350 records — Used for epoch evaluation, learning rate monitoring, and best-checkpoint selection.
- **Locked Test Set (`phinc_test.jsonl`):** 1,351 records — Cryptographically locked on local workstation; **excluded entirely from this bundle**.
- **Excluded Records:** 235 low-quality or corrupt records quarantined during Step 34B cleaning.

### 3. Translation Task Formulation
We formulate translation as text-to-text sequence generation with a standardized task prefix:
$$\text{Source Input: } \texttt{"translate Hinglish to English: " } + \text{source\_text}$$
$$\text{Target Output: } \text{target\_text}$$


---
## SECTION B — Runtime Setup and Hardware Verification

Google Colab provides access to GPU runtimes. For reproducible fine-tuning of `google/mt5-small`, an **NVIDIA T4 GPU** (16GB VRAM) or better is required.

> **Colab Setup Instructions:**  
> Go to menu: **Runtime** $\rightarrow$ **Change runtime type** $\rightarrow$ Select **T4 GPU** (or any GPU) $\rightarrow$ Click **Save**.


In [ ]:
# Cell B1: Inspect Python, PyTorch, and CUDA GPU Hardware
import os
import sys
import torch

print("=" * 70)
print("CODEMIX INTELLIGENCE — RUNTIME HARDWARE INSPECTION")
print("=" * 70)
print(f"Python Executable: {sys.executable}")
print(f"Python Version:    {sys.version.split()[0]}")
print(f"PyTorch Version:   {torch.__version__}")
print(f"CUDA Available:    {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CRITICAL ERROR: CUDA GPU is not available!\n"
        "Please select a GPU runtime: Runtime -> Change runtime type -> T4 GPU -> Save."
    )

gpu_count = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
total_vram_gb = gpu_props.total_memory / (1024 ** 3)
cuda_version = torch.version.cuda
bf16_supported = torch.cuda.is_bf16_supported()

print(f"Detected GPUs:     {gpu_count}")
print(f"Active Device [0]: {gpu_name}")
print(f"Compute Capability:{gpu_props.major}.{gpu_props.minor}")
print(f"Total GPU VRAM:    {total_vram_gb:.2f} GB")
print(f"CUDA Version:      {cuda_version}")
print(f"BF16 Supported:    {bf16_supported}")
print("=" * 70)
print("[PASS] GPU runtime successfully verified. Ready for training.")


In [ ]:
# Cell B2: Install Compatible Dependencies with Strict Constraints
# We enforce tested version ranges to ensure stable Seq2SeqTrainer and SentencePiece operation.
!pip install --quiet "transformers>=5.0.0,<6.0.0" "accelerate>=1.0.0,<2.0.0" "sentencepiece>=0.2.0,<1.0.0" "sacrebleu>=2.4.0,<3.0.0" "safetensors>=0.4.0" "datasets>=2.18.0" "protobuf>=3.20.0"

import transformers
import accelerate
import sentencepiece
import sacrebleu
import safetensors

print("=" * 70)
print("RESOLVED PACKAGE VERSIONS:")
print("=" * 70)
print(f"  transformers:  {transformers.__version__}")
print(f"  accelerate:    {accelerate.__version__}")
print(f"  sentencepiece: {sentencepiece.__version__}")
print(f"  sacrebleu:     {sacrebleu.__version__}")
print(f"  safetensors:   {safetensors.__version__}")
print("=" * 70)
print("[PASS] All required packages installed and imported successfully.")


In [ ]:
# Cell B3: Set Global Reproducibility Seeds
import random
import numpy as np
import torch

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"[OK] Global random seed set to {seed} (Python, NumPy, PyTorch, CUDA).")

set_seed(SEED)


---
## SECTION C — Google Drive Integration and Storage Hierarchy

We mount Google Drive to persist all training checkpoints, evaluation metrics, logs, and the final exported model package. If the runtime disconnects, your training progress is preserved.

### Directory Layout in Drive:
```
/content/drive/MyDrive/CodeMix_Intelligence/
  ├── mt5_phinc_training_bundle.zip          <-- Input bundle
  └── mt5_phinc_baseline/
      ├── checkpoints/                       <-- Saved epoch checkpoints (max 2)
      ├── best_model/                        <-- Selected best validation model
      ├── metrics/                           <-- Loss, BLEU, chrF histories
      ├── logs/                              <-- Training run logs
      ├── metadata/                          <-- Manifest, config, and model card
      └── mt5_phinc_baseline_best_model.zip  <-- Final downloadable model package
```


In [ ]:
# Cell C1: Mount Google Drive and Establish Storage Hierarchy
from google.colab import drive
import os

print("Mounting Google Drive to /content/drive...")
drive.mount('/content/drive')

BASE_DRIVE_DIR = "/content/drive/MyDrive/CodeMix_Intelligence/mt5_phinc_baseline"
DIRECTORIES = {
    "base": BASE_DRIVE_DIR,
    "checkpoints": os.path.join(BASE_DRIVE_DIR, "checkpoints"),
    "best_model": os.path.join(BASE_DRIVE_DIR, "best_model"),
    "metrics": os.path.join(BASE_DRIVE_DIR, "metrics"),
    "logs": os.path.join(BASE_DRIVE_DIR, "logs"),
    "metadata": os.path.join(BASE_DRIVE_DIR, "metadata")
}

for name, path in DIRECTORIES.items():
    os.makedirs(path, exist_ok=True)
    print(f"[OK] Directory ready: {name:12s} -> {path}")

print("=" * 70)
print("[PASS] Persistent Google Drive storage hierarchy established.")


In [ ]:
# Cell C2: Training Bundle Ingestion and Checksum Verification
import os
import shutil
import hashlib
from google.colab import files

# Authoritative cryptographic constants verified during Step 34D preflight
EXPECTED_BUNDLE_SHA256 = "d2af7d5a193b641878f34b19f1a0dab5dd1ff65cc7c78c87f76884e31b87ff60"
EXPECTED_BUNDLE_SIZE = 1387072

drive_bundle_path = "/content/drive/MyDrive/CodeMix_Intelligence/mt5_phinc_training_bundle.zip"
local_bundle_path = "/content/mt5_phinc_training_bundle.zip"

def compute_sha256(path):
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(65536):
            sha.update(chunk)
    return sha.hexdigest()

print("=" * 70)
print("TRAINING BUNDLE INGESTION")
print("=" * 70)

if os.path.exists(drive_bundle_path):
    print(f"Found training bundle in Google Drive: {drive_bundle_path}")
    shutil.copy2(drive_bundle_path, local_bundle_path)
else:
    print(f"Bundle not found at Drive location: {drive_bundle_path}")
    print("Please upload 'mt5_phinc_training_bundle.zip' via the upload prompt below:")
    uploaded = files.upload()
    uploaded_name = list(uploaded.keys())[0]
    shutil.move(uploaded_name, local_bundle_path)
    # Save to Drive for future sessions
    os.makedirs(os.path.dirname(drive_bundle_path), exist_ok=True)
    shutil.copy2(local_bundle_path, drive_bundle_path)
    print(f"Saved uploaded bundle to Google Drive: {drive_bundle_path}")

actual_bundle_sha = compute_sha256(local_bundle_path)
actual_bundle_size = os.path.getsize(local_bundle_path)

print(f"Bundle File:   {local_bundle_path}")
print(f"Bundle Size:   {actual_bundle_size:,} bytes (Expected: {EXPECTED_BUNDLE_SIZE:,})")
print(f"Bundle SHA256: {actual_bundle_sha}")

if actual_bundle_sha != EXPECTED_BUNDLE_SHA256:
    raise ValueError(
        f"CRITICAL SECURITY / INTEGRITY FAULT: Training bundle SHA-256 mismatch!\n"
        f"  Actual:   {actual_bundle_sha}\n"
        f"  Expected: {EXPECTED_BUNDLE_SHA256}\n"
        "STOPPING. The bundle file is corrupted or unauthorized. Re-upload original."
    )

print("=" * 70)
print("[PASS] Training bundle cryptographically verified bitwise.")


---
## SECTION D — Bundle Extraction & Pre-Training Dataset Verification

We extract the verified ZIP archive to local Colab storage (`/content/CodeMix_Intelligence_mt5_phinc_data`) and verify:
1. Exact member files: strictly `phinc_train.jsonl`, `phinc_validation.jsonl`, and `training_bundle_manifest.json`.
2. Absolute absence of test data.
3. Record counts: 10,802 training rows, 1,350 validation rows.
4. Bitwise SHA-256 matching the pre-split manifests.
5. Zero ID overlap between splits and non-empty text fields.


In [ ]:
# Cell D1: Extract Bundle and Verify Archive Members
import zipfile
import json

DATA_DIR = "/content/CodeMix_Intelligence_mt5_phinc_data"
os.makedirs(DATA_DIR, exist_ok=True)

with zipfile.ZipFile(local_bundle_path, "r") as zf:
    namelist = zf.namelist()
    print("Archive members detected:", namelist)
    
    # 1. Strict member count and identity check
    assert len(namelist) == 3, f"Security Violation: Expected exactly 3 files, found {len(namelist)}"
    expected_members = {"phinc_train.jsonl", "phinc_validation.jsonl", "training_bundle_manifest.json"}
    assert set(namelist) == expected_members, f"Security Violation: Unexpected files {set(namelist) - expected_members}"
    
    # 2. Strict anti-test check
    for name in namelist:
        assert "test" not in name.lower(), f"CRITICAL LEAKAGE DETECTED: Test file in training bundle: {name}"
        
    zf.extractall(DATA_DIR)

print(f"[PASS] Archive cleanly extracted to: {DATA_DIR}")


In [ ]:
# Cell D2: Verify Split Checksums, Record Counts, and ID Disjointness
train_file = os.path.join(DATA_DIR, "phinc_train.jsonl")
val_file = os.path.join(DATA_DIR, "phinc_validation.jsonl")
manifest_file = os.path.join(DATA_DIR, "training_bundle_manifest.json")

EXPECTED_TRAIN_SHA = "f9b9b4e5678c5ce628891d0fa9e53224d4723a8015828a95206ce2a3c34f7f41"
EXPECTED_VAL_SHA = "b63971bbc05c692279f092c682d6c2bd1e184fe759009a3a52576c5e54500007"

actual_train_sha = compute_sha256(train_file)
actual_val_sha = compute_sha256(val_file)

assert actual_train_sha == EXPECTED_TRAIN_SHA, f"Train SHA mismatch! Got: {actual_train_sha}"
assert actual_val_sha == EXPECTED_VAL_SHA, f"Val SHA mismatch! Got: {actual_val_sha}"

# Load and validate records
train_records = []
with open(train_file, "r", encoding="utf-8") as f:
    for line in f:
        train_records.append(json.loads(line))

val_records = []
with open(val_file, "r", encoding="utf-8") as f:
    for line in f:
        val_records.append(json.loads(line))

assert len(train_records) == 10802, f"Train count mismatch: {len(train_records)} (Expected 10,802)"
assert len(val_records) == 1350, f"Val count mismatch: {len(val_records)} (Expected 1,350)"

# Verify required fields and non-empty text
for split_name, split_data in [("train", train_records), ("validation", val_records)]:
    for idx, item in enumerate(split_data):
        assert "source_text" in item and len(item["source_text"].strip()) > 0, f"Empty source_text in {split_name} at index {idx}"
        assert "target_text" in item and len(item["target_text"].strip()) > 0, f"Empty target_text in {split_name} at index {idx}"
        assert "example_id" in item, f"Missing example_id in {split_name} at index {idx}"

# Verify zero ID overlap between train and validation
train_ids = set(r["example_id"] for r in train_records)
val_ids = set(r["example_id"] for r in val_records)
overlap = train_ids.intersection(val_ids)
assert len(overlap) == 0, f"DATA LEAKAGE: Overlapping IDs between train and val: {overlap}"

# Inspect manifest
with open(manifest_file, "r", encoding="utf-8") as f:
    bundle_manifest = json.load(f)

print("=" * 70)
print("SPLIT INTEGRITY VERIFIED:")
print("=" * 70)
print(f"  Training Records:   {len(train_records):,} (SHA-256 Verified)")
print(f"  Validation Records: {len(val_records):,} (SHA-256 Verified)")
print(f"  Unique Train IDs:   {len(train_ids):,}")
print(f"  Unique Val IDs:     {len(val_ids):,}")
print(f"  Split ID Overlap:   {len(overlap)} (Zero leakage)")
print(f"  Bundle Purpose:     {bundle_manifest.get('purpose')}")
print(f"  Test Protection:    {bundle_manifest.get('locked_test_exclusion_statement')}")
print("=" * 70)
print("[PASS] All dataset splits verified bitwise and structurally.")


---
## SECTION E — Load Pretrained `google/mt5-small` and Tokenizer

We download Google's official pretrained `google/mt5-small` sequence-to-sequence model and its SentencePiece multilingual tokenizer directly from the Hugging Face Hub.

### Model Architecture Details:
- **Base Model:** `google/mt5-small`
- **Architecture:** Multilingual T5 (Encoder-Decoder)
- **Pretrained Corpus:** mC4 (101 languages, including Hindi and English)
- **Parameters:** ~300 Million
- **Vocabulary:** 250,100 SentencePiece subword tokens


In [ ]:
# Cell E1: Download and Load Pretrained Model & Tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/mt5-small"

print(f"Loading SentencePiece tokenizer for: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

print(f"Loading Seq2Seq transformer weights for: {MODEL_NAME}...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 70)
print("MODEL ARCHITECTURE & VOCABULARY SUMMARY:")
print("=" * 70)
print(f"  Model Type:             {model.config.model_type.upper()}")
print(f"  Total Parameters:       {total_params:,}")
print(f"  Trainable Parameters:   {trainable_params:,}")
print(f"  Vocabulary Size:        {len(tokenizer):,} tokens")
print(f"  Hidden Size (d_model):  {model.config.d_model}")
print(f"  Encoder Layers / Heads: {model.config.num_layers} layers / {model.config.num_heads} heads")
print(f"  Decoder Layers / Heads: {model.config.num_decoder_layers} layers / {model.config.num_heads} heads")
print(f"  Feed-Forward Dim:       {model.config.d_ff}")
print("=" * 70)
print("[PASS] Pretrained mT5-small model and tokenizer successfully loaded.")


---
## SECTION F — Translation Task Prefix and Prompt Formatting

T5 and mT5 are text-to-text models that utilize task prefixes to condition generation. For our Romanized Hinglish-to-English translation baseline, we enforce the canonical task prefix:
$$\texttt{"translate Hinglish to English: "}$$

### Formatting Specification:
- **Source Input:** `"translate Hinglish to English: " + source_text`
- **Target Output:** `target_text` (No prefix is added to targets)
- **Application Rule:** This exact prefix is frozen and saved to `task_config.json` for all future backend inference and API serving.


In [ ]:
# Cell F1: Define and Validate Task Prefix Formatting
TASK_PREFIX = "translate Hinglish to English: "

# Display demonstration samples
print("=" * 70)
print("TASK PREFIX FORMATTING EXAMPLES:")
print("=" * 70)
for i in [0, 10, 42]:
    ex = train_records[i]
    formatted_input = TASK_PREFIX + ex["source_text"]
    target_output = ex["target_text"]
    print(f"Example {i+1} [ID: {ex['example_id']}]:")
    print(f"  Raw Source:   {repr(ex['source_text'])}")
    print(f"  Prefix Input: {repr(formatted_input)}")
    print(f"  Target Gold:  {repr(target_output)}")
    print("-" * 70)

print("[PASS] Task prefix established and verified.")


---
## SECTION G — Tokenization Preflight Audit & Truncation Safeguard

Before defining dataset loaders, we measure the exact SentencePiece token lengths across all 10,802 training and 1,350 validation examples. We must ensure that our chosen maximum sequence length results in **zero silent truncation**.


In [ ]:
# Cell G1: Comprehensive Tokenization Preflight Audit
import numpy as np

all_records = train_records + val_records

src_token_lengths = []
tgt_token_lengths = []

print(f"Running tokenization preflight across all {len(all_records):,} records...")
for rec in all_records:
    src_text = TASK_PREFIX + rec["source_text"]
    tgt_text = rec["target_text"]
    
    src_tokens = tokenizer(src_text, truncation=False)["input_ids"]
    tgt_tokens = tokenizer(tgt_text, truncation=False)["input_ids"]
    
    src_token_lengths.append(len(src_tokens))
    tgt_token_lengths.append(len(tgt_tokens))

max_src = int(np.max(src_token_lengths))
med_src = float(np.median(src_token_lengths))
p95_src = float(np.percentile(src_token_lengths, 95))
p99_src = float(np.percentile(src_token_lengths, 99))

max_tgt = int(np.max(tgt_token_lengths))
med_tgt = float(np.median(tgt_token_lengths))
p95_tgt = float(np.percentile(tgt_token_lengths, 95))
p99_tgt = float(np.percentile(tgt_token_lengths, 99))

print("=" * 70)
print("SENTENCEPIECE TOKEN LENGTH AUDIT:")
print("=" * 70)
print(f"Source (with Prefix): Max = {max_src:3d} | Median = {med_src:4.1f} | P95 = {p95_src:4.1f} | P99 = {p99_src:4.1f}")
print(f"Target (English):     Max = {max_tgt:3d} | Median = {med_tgt:4.1f} | P95 = {p95_tgt:4.1f} | P99 = {p99_tgt:4.1f}")
print("=" * 70)

# Establish maximum sequence lengths
# Baseline candidate: 128. If max length exceeds 128, round up to next multiple of 8, capped at 256.
def compute_safe_length(measured_max, candidate=128, cap=256):
    if measured_max <= candidate:
        return candidate
    # Round up to next multiple of 8
    safe_len = int(np.ceil(measured_max / 8.0) * 8)
    return min(safe_len, cap)

MAX_SOURCE_LENGTH = compute_safe_length(max_src, candidate=128, cap=256)
MAX_TARGET_LENGTH = compute_safe_length(max_tgt, candidate=128, cap=256)

truncated_src = sum(1 for l in src_token_lengths if l > MAX_SOURCE_LENGTH)
truncated_tgt = sum(1 for l in tgt_token_lengths if l > MAX_TARGET_LENGTH)

print(f"Selected MAX_SOURCE_LENGTH: {MAX_SOURCE_LENGTH} (Truncated examples: {truncated_src} / {len(all_records)})")
print(f"Selected MAX_TARGET_LENGTH: {MAX_TARGET_LENGTH} (Truncated examples: {truncated_tgt} / {len(all_records)})")

if truncated_src > 0 or truncated_tgt > 0:
    print(f"[WARNING] Minor truncation detected: {truncated_src} source, {truncated_tgt} target.")
else:
    print("[PASS] ZERO truncation verified across all training and validation examples.")


---
## SECTION H — Dataset Objects and Data Collator

We implement a PyTorch `Dataset` that yields tokenized encoder inputs (`input_ids`, `attention_mask`) and decoder targets (`labels`).

### Key Implementation Safeguards:
1. **Dynamic Padding:** We leave samples unpadded in `__getitem__`. The `DataCollatorForSeq2Seq` dynamically pads each batch to the longest sequence in that specific mini-batch, saving significant GPU memory and accelerating training.
2. **Loss Masking with `-100`:** Target padding tokens are assigned label `-100`, which PyTorch's `CrossEntropyLoss` ignores during backpropagation.
3. **Strict Isolation:** Only `train_records` and `val_records` are ingested.


In [ ]:
# Cell H1: Define Seq2Seq Dataset Class and Instantiate Data Collator
from torch.utils.data import Dataset
from transformers import DataCollatorForSeq2Seq

class HinglishTranslationDataset(Dataset):
    def __init__(self, records, tokenizer, prefix, max_src_len, max_tgt_len):
        self.records = records
        self.tokenizer = tokenizer
        self.prefix = prefix
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        src_text = self.prefix + rec["source_text"]
        tgt_text = rec["target_text"]

        # Tokenize source sentence (dynamic padding handled by collator)
        model_inputs = self.tokenizer(
            src_text,
            max_length=self.max_src_len,
            padding=False,
            truncation=True
        )

        # Tokenize target sentence
        labels = self.tokenizer(
            text_target=tgt_text,
            max_length=self.max_tgt_len,
            padding=False,
            truncation=True
        )

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

# Instantiate datasets
train_dataset = HinglishTranslationDataset(
    train_records, tokenizer, TASK_PREFIX, MAX_SOURCE_LENGTH, MAX_TARGET_LENGTH
)
val_dataset = HinglishTranslationDataset(
    val_records, tokenizer, TASK_PREFIX, MAX_SOURCE_LENGTH, MAX_TARGET_LENGTH
)

# Dynamic collator for Seq2Seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

print(f"[OK] Training Dataset ready:   {len(train_dataset):,} samples.")
print(f"[OK] Validation Dataset ready: {len(val_dataset):,} samples.")
print(f"[OK] DataCollator ready with dynamic batch padding and -100 label masking.")


---
## SECTION I — Generation-Based Evaluation Metrics (SacreBLEU & chrF)

During model training, validation quality is measured using **SacreBLEU** and **chrF++** on decoded generated English text.
- **SacreBLEU:** Standard benchmark metric measuring n-gram precision with brevity penalty.
- **chrF:** Character n-gram F-score, particularly suited for translation evaluation.
- **Strict Rule:** We do not compute classification accuracy for sentence translation, and BLEU is never described as "percentage accuracy".


In [ ]:
# Cell I1: Implement Generation-Based Evaluation Function
import sacrebleu
import numpy as np

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Decode predicted token IDs into text
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100 in labels with pad_token_id to enable decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Normalize whitespace
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    # Compute SacreBLEU score
    bleu_metric = sacrebleu.corpus_bleu(decoded_preds, decoded_labels)
    
    # Compute chrF score
    chrf_metric = sacrebleu.corpus_chrf(decoded_preds, decoded_labels)

    return {
        "bleu": round(bleu_metric.score, 2),
        "chrf": round(chrf_metric.score, 2)
    }

# Record metric version and signature
dummy_bleu = sacrebleu.corpus_bleu(["This is a test"], [["This is a test"]])
sacrebleu_version = sacrebleu.__version__
sacrebleu_signature = str(dummy_bleu)

print("=" * 70)
print("METRIC ENGINE CONFIGURATION:")
print("=" * 70)
print(f"  SacreBLEU Version:   {sacrebleu_version}")
print(f"  SacreBLEU Signature: {sacrebleu_signature}")
print("=" * 70)
print("[PASS] Generation-based evaluation metric verified.")


---
## SECTION J — Training Configuration & Hyperparameters

We configure conservative, memory-safe hyperparameters suited for fine-tuning `mT5-small` on a single NVIDIA T4 GPU:
- **Batch Size:** 4 per device $\times$ 4 gradient accumulation steps = **Effective Batch Size 16**.
- **Learning Rate:** $1 \times 10^{-4}$ with linear warmup and decay.
- **Epochs:** 3 full epochs.
- **Best Model Metric:** Validation SacreBLEU (`bleu`).
- **Precision:** FP16 mixed precision (BF16 enabled only if hardware natively supports it; never both).
- **API Version Compatibility:** We inspect `Seq2SeqTrainingArguments` parameter signatures dynamically to support both `evaluation_strategy` (v4.x) and `eval_strategy` (v5.x).


In [ ]:
# Cell J1: Configure Seq2SeqTrainingArguments with Version-Proof Inspection
import inspect
from transformers import Seq2SeqTrainingArguments

sig = inspect.signature(Seq2SeqTrainingArguments.__init__)

# Determine version-compatible argument names
eval_strategy_arg = "eval_strategy" if "eval_strategy" in sig.parameters else "evaluation_strategy"
supports_save_only_model = "save_only_model" in sig.parameters

# Precision selection: BF16 if supported, else FP16 on CUDA
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

args_dict = {
    "output_dir": DIRECTORIES["checkpoints"],
    "overwrite_output_dir": True,
    "num_train_epochs": 3,
    "learning_rate": 1e-4,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,  # Effective Batch Size = 16
    "per_device_eval_batch_size": 4,
    "lr_scheduler_type": "linear",
    "max_grad_norm": 1.0,
    "logging_strategy": "steps",
    "logging_steps": 50,
    eval_strategy_arg: "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "bleu",
    "greater_is_better": True,
    "predict_with_generate": True,
    "generation_max_length": MAX_TARGET_LENGTH,
    "generation_num_beams": 4,
    "fp16": use_fp16,
    "bf16": use_bf16,
    "report_to": "none",
    "seed": SEED
}

if supports_save_only_model:
    # Save model weights without full optimizer state to conserve Drive storage
    args_dict["save_only_model"] = True

training_args = Seq2SeqTrainingArguments(**args_dict)

print("=" * 70)
print("TRAINING HYPERPARAMETERS & RUNTIME CONFIGURATION:")
print("=" * 70)
print(f"  Base Model:                 {MODEL_NAME}")
print(f"  Random Seed:                {SEED}")
print(f"  Training Epochs:            {training_args.num_train_epochs}")
print(f"  Learning Rate:              {training_args.learning_rate}")
print(f"  Weight Decay:               {training_args.weight_decay}")
print(f"  Warmup Ratio:               {training_args.warmup_ratio}")
print(f"  Per-Device Train Batch:     {training_args.per_device_train_batch_size}")
print(f"  Gradient Accumulation:      {training_args.gradient_accumulation_steps}")
print(f"  Effective Batch Size:       {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Per-Device Eval Batch:      {training_args.per_device_eval_batch_size}")
print(f"  Mixed Precision:            FP16={training_args.fp16}, BF16={training_args.bf16}")
print(f"  Evaluation Cadence:         {getattr(training_args, eval_strategy_arg)}")
print(f"  Checkpoint Save Cadence:    {training_args.save_strategy}")
print(f"  Max Checkpoints Saved:      {training_args.save_total_limit}")
print(f"  Metric for Best Model:      {training_args.metric_for_best_model}")
print(f"  Generation Num Beams:       {training_args.generation_num_beams}")
print(f"  Generation Max Length:      {training_args.generation_max_length}")
print(f"  Save Only Model:            {getattr(training_args, 'save_only_model', False)}")
print("=" * 70)
print("[PASS] Seq2SeqTrainingArguments successfully initialized.")


---
## SECTION K — Checkpointing and Drive Storage Architecture

Checkpoints are written directly to Google Drive:
`/content/drive/MyDrive/CodeMix_Intelligence/mt5_phinc_baseline/checkpoints/`

### Storage Safeguards:
1. `save_total_limit = 2`: At most two checkpoints are retained, preventing Colab / Google Drive storage exhaustion.
2. `load_best_model_at_end = True`: When training completes, the trainer automatically restores the checkpoint with the highest validation SacreBLEU.
3. Isolated from MuRIL: All mT5 checkpoints reside inside `mt5_phinc_baseline/` and never touch `artifacts/models/muril_lid_baseline/`.


---
## SECTION L — Seq2SeqTrainer Initialization & Training Execution

We instantiate `Seq2SeqTrainer` with an `EarlyStoppingCallback` (patience = 2 epochs). Evaluation runs at the end of each epoch with beam search decoding (`num_beams = 4`).


In [ ]:
# Cell L1: Initialize Seq2SeqTrainer
from transformers import Seq2SeqTrainer, EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("[PASS] Seq2SeqTrainer initialized with generation-based validation & early stopping.")


In [ ]:
# Cell L2: Execute Training Loop
import time

print("=" * 70)
print("STARTING mT5-SMALL HINGLISH TRANSLATION FINE-TUNING")
print("=" * 70)
start_time = time.time()

train_result = trainer.train()

training_runtime_sec = time.time() - start_time
print("=" * 70)
print("TRAINING COMPLETED!")
print(f"  Total Runtime:       {training_runtime_sec / 60:.2f} minutes")
print(f"  Global Steps:        {train_result.global_step}")
print(f"  Final Training Loss: {train_result.training_loss:.4f}")
print("=" * 70)


---
## SECTION M — Save Final Best Model to Google Drive

After training completes and the best validation checkpoint is loaded, we export the complete model and SentencePiece tokenizer to:
`/content/drive/MyDrive/CodeMix_Intelligence/mt5_phinc_baseline/best_model/`


In [ ]:
# Cell M1: Save Best Model, Tokenizer, and Task Metadata
best_model_dir = DIRECTORIES["best_model"]
print(f"Saving final best model weights and tokenizer to: {best_model_dir}...")

trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)

# Save task configuration metadata for future backend integration
task_config = {
    "task": "Romanized Hinglish to English Translation",
    "task_prefix": TASK_PREFIX,
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "generation_num_beams": 4,
    "base_model": MODEL_NAME,
    "dataset": "PHINC (Srivastava & Singh, W-NUT 2020)",
    "license": "CC BY 4.0",
    "train_records": len(train_records),
    "validation_records": len(val_records),
    "locked_test_records": 1351
}

with open(os.path.join(best_model_dir, "task_config.json"), "w", encoding="utf-8") as f:
    json.dump(task_config, f, indent=2)

# Verify reloadability from disk
print("Verifying model reloadability from saved directory...")
test_reload = AutoModelForSeq2SeqLM.from_pretrained(best_model_dir)
test_tok_reload = AutoTokenizer.from_pretrained(best_model_dir, use_fast=False)
del test_reload
del test_tok_reload

print(f"[PASS] Model and tokenizer saved and reloadability verified at: {best_model_dir}")


---
## SECTION N — Training Metrics, Environment Metadata, and Model Card

We compile and persist complete training histories, validation evaluations, environment metadata, and the official model card to Google Drive.


In [ ]:
# Cell N1: Export Metrics, Metadata, and Model Card
import datetime

# 1. Final evaluation on validation set
print("Running final validation evaluation...")
val_eval = trainer.evaluate(eval_dataset=val_dataset)
print(f"Final Validation Metrics: {val_eval}")

# 2. Save metrics
with open(os.path.join(DIRECTORIES["metrics"], "validation_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(val_eval, f, indent=2)

with open(os.path.join(DIRECTORIES["metrics"], "training_history.json"), "w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, indent=2)

with open(os.path.join(DIRECTORIES["metrics"], "best_checkpoint_info.json"), "w", encoding="utf-8") as f:
    json.dump({
        "best_model_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "metric_name": training_args.metric_for_best_model
    }, f, indent=2)

# 3. Save environment metadata
env_metadata = {
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "python_version": sys.version,
    "pytorch_version": torch.__version__,
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
    "gpu_name": gpu_name if torch.cuda.is_available() else "None",
    "total_vram_gb": total_vram_gb if torch.cuda.is_available() else 0,
    "transformers_version": transformers.__version__,
    "accelerate_version": accelerate.__version__,
    "sentencepiece_version": sentencepiece.__version__,
    "sacrebleu_version": sacrebleu.__version__,
    "sacrebleu_signature": sacrebleu_signature
}
with open(os.path.join(DIRECTORIES["metadata"], "environment.json"), "w", encoding="utf-8") as f:
    json.dump(env_metadata, f, indent=2)

# 4. Save training configuration
with open(os.path.join(DIRECTORIES["metadata"], "training_config.json"), "w", encoding="utf-8") as f:
    json.dump(args_dict, f, indent=2, default=str)

# 5. Generate Model Card
model_card_content = f"""# Model Card: CodeMix Intelligence mT5-small Hinglish-to-English Baseline

## Model Overview
- **Model Name:** CodeMix Intelligence mT5-small Baseline (v2-alpha)
- **Base Pretrained Model:** `google/mt5-small`
- **Architecture:** Multilingual T5 (Seq2Seq Transformer, 6 encoder layers, 6 decoder layers, 8 heads, d_model=512)
- **Task:** Romanized Hinglish Code-Mixed Sentence to Natural English Translation
- **Input Task Prefix:** `translate Hinglish to English: `
- **Dataset:** PHINC (Parallel Hinglish Social Media Corpus, W-NUT 2020)
- **License:** Creative Commons Attribution 4.0 International (CC BY 4.0)
- **Translation Provenance:** human_translated
- **Training Records:** 10,802 sentence pairs
- **Validation Records:** 1,350 sentence pairs
- **Locked Test Records:** 1,351 sentence pairs (Cryptographically locked on local workstation; zero leakage)

## Training Configuration
- **Hardware:** {gpu_name} ({total_vram_gb:.2f} GB VRAM)
- **Effective Batch Size:** 16 (per-device batch 4 x 4 grad accum)
- **Learning Rate:** 1e-4 (linear warmup 0.06, linear decay)
- **Epochs:** 3 full epochs
- **Precision:** {'BF16' if use_bf16 else ('FP16' if use_fp16 else 'FP32')}
- **Best Model Selection Metric:** Validation SacreBLEU

## Final Validation Performance (Validation Set Only)
- **Validation SacreBLEU:** {val_eval.get('eval_bleu', 'N/A')}
- **Validation chrF++:**   {val_eval.get('eval_chrf', 'N/A')}
- **Validation Loss:**     {val_eval.get('eval_loss', 'N/A')}
- **Best Checkpoint:**     {trainer.state.best_model_checkpoint}

## Scope and Limitations
1. **Script Scope:** Fine-tuned primarily on Romanized Hinglish (Latin script).
2. **Language Scope:** Does NOT support Telugu-to-English translation.
3. **Devanagari Coverage:** Minimal Devanagari coverage; do not use for pure Hindi script translation.
4. **Test Set Status:** Test metrics are intentionally NOT reported here. Final evaluation occurs independently on the locked test set.
"""

with open(os.path.join(DIRECTORIES["metadata"], "model_card.md"), "w", encoding="utf-8") as f:
    f.write(model_card_content)

print("[PASS] All metrics, environment metadata, and model card saved to Drive.")


---
## SECTION O — Test Set Protection Notice

> [!IMPORTANT]
> **Strict Anti-Leakage Protocol:**
> In compliance with the scientific evaluation policy of CodeMix Intelligence, **no test evaluation cell is provided in this notebook**.
> 
> The official test split (`phinc_test.jsonl`, 1,351 records) remains cryptographically locked on the local development workstation. Evaluation on the locked test split will occur only after the trained model package is imported back to the local repository, ensuring zero data leakage or test set overfitting during model development.


---
## SECTION P — Best Model Archive Export

We compress the saved model and tokenizer directory into a portable archive for download:  
`/content/drive/MyDrive/CodeMix_Intelligence/mt5_phinc_baseline/mt5_phinc_baseline_best_model.zip`


In [ ]:
# Cell P1: Create Downloadable Best Model ZIP Archive and Verify Integrity
import shutil
import zipfile

zip_target = os.path.join(DIRECTORIES["base"], "mt5_phinc_baseline_best_model.zip")
print(f"Compressing best model directory: {best_model_dir} -> {zip_target}...")

shutil.make_archive(
    base_name=zip_target.replace(".zip", ""),
    format="zip",
    root_dir=best_model_dir
)

zip_sha256 = compute_sha256(zip_target)
zip_size = os.path.getsize(zip_target)

# Verify ZIP member integrity
with zipfile.ZipFile(zip_target, "r") as zf:
    members = zf.namelist()
    print("Archive members:", members)
    assert any("model" in m or "safetensors" in m or "bin" in m for m in members), "Missing model weights!"
    assert any("spiece" in m or "tokenizer" in m for m in members), "Missing tokenizer files!"

print("=" * 70)
print("FINAL MODEL ARCHIVE EXPORT:")
print("=" * 70)
print(f"  Archive Path:    {zip_target}")
print(f"  Archive Size:    {zip_size:,} bytes ({zip_size / (1024**2):.2f} MB)")
print(f"  Archive SHA-256: {zip_sha256}")
print("=" * 70)
print("[PASS] Portable model ZIP archive successfully created and verified.")


---
## SECTION Q — Interactive Inference Demonstration

Test the fine-tuned model interactively with sample Romanized Hinglish sentences.


In [ ]:
# Cell Q1: Interactive Translation Demonstration
def translate_hinglish(sentence: str) -> str:
    prompt = TASK_PREFIX + sentence.strip()
    inputs = tokenizer(prompt, return_tensors="pt", max_length=MAX_SOURCE_LENGTH, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=MAX_TARGET_LENGTH,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

demo_samples = [
    "Aap kaisa feel kar rahe ho abhi?",
    "Kal office jaldi aana padega kyunki important meeting hai.",
    "Mujhe yeh movie bilkul pasand nahi aayi.",
    "Bhai please mujhe project me help kardo."
]

print("=" * 70)
print("INTERACTIVE HINGLISH-TO-ENGLISH TRANSLATION TEST:")
print("=" * 70)
for sample in demo_samples:
    trans = translate_hinglish(sample)
    print(f"Hinglish:    {sample}")
    print(f"Translation: {trans}")
    print("-" * 70)
